# Muse EEG Heads — CBraMod + Head A smoke

Frozen **CBraMod** + tiny **Head A** on Sleep-EDF SC4001 N1-slice Muse-proxy windows.

**Task (smoke):** binary `drowsy` vs `hypnagogic` (W / N1).

**Input adapter:** `(B,4,T)` @ 256 Hz → resample 200 Hz → `(B,4,n_patches,200)`. Keep C=4 (no fabricated 10–20 montage). See `docs/cbramod_notes.md`.

**Data:** `windwerfer/muse-eeg-heads-src` + `windwerfer/muse-eeg-heads-cache` (or prebuilt `muse-eeg-heads-windows` if attached).

**Licenses:** Sleep-EDF PhysioNet **ODC-By**; CBraMod **Apache-2.0**. NO LUNA / L-FAME / SEED-VIG.


## 1. Setup


In [ ]:
# Setup — helpers from muse-eeg-heads-src; prefer uv
import os, sys, random, shutil, subprocess, json, hashlib
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone

WORKING = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".").resolve()
INPUT_ROOT = Path("/kaggle/input")

def _list_tree(root: Path, max_depth=3):
    out = []
    if not root.exists():
        return out
    for p in sorted(root.rglob("*")):
        try:
            rel = p.relative_to(root)
        except Exception:
            continue
        if len(rel.parts) <= max_depth:
            out.append(str(rel) + ("/" if p.is_dir() else ""))
    return out[:80]

print("kaggle input tree:", _list_tree(INPUT_ROOT))

def find_src_dir():
    """Locate muse-eeg-heads-src across flat or nested /kaggle/input layouts."""
    names = ("muse-eeg-heads-src",)
    if INPUT_ROOT.exists():
        for name in names:
            direct = INPUT_ROOT / name
            if direct.exists() and list(direct.glob("*.py")):
                return direct
        # nested e.g. /kaggle/input/datasets/.../muse-eeg-heads-src
        for p in INPUT_ROOT.rglob("sleep_edf.py"):
            return p.parent
        for name in names:
            hits = [p for p in INPUT_ROOT.rglob(name) if p.is_dir()]
            for h in hits:
                if list(h.glob("*.py")):
                    return h
    local = Path("/workspace/muse-eeg-heads/src")
    if local.exists() and list(local.glob("*.py")):
        return local
    return None

src_dir = find_src_dir()
SRC = WORKING / "src"
SRC.mkdir(parents=True, exist_ok=True)
if src_dir is None:
    raise FileNotFoundError(
        "Missing muse-eeg-heads-src helpers. Add windwerfer/muse-eeg-heads-src. "
        f"tree={_list_tree(INPUT_ROOT)}"
    )
for f in src_dir.glob("*.py"):
    shutil.copy(f, SRC / f.name)
print("loaded modules from", src_dir, "→", SRC)

sys.path.insert(0, str(WORKING))

need = []
for mod, pipname in [("numpy", "numpy"), ("torch", "torch"), ("scipy", "scipy")]:
    try:
        __import__(mod)
    except ImportError:
        need.append(pipname)
if need:
    if subprocess.call(["bash", "-lc", "command -v uv >/dev/null"]) == 0:
        subprocess.check_call(["uv", "pip", "install", "--system", "-q", *need])
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *need])

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("ROOT", WORKING, "torch", torch.__version__)


## 2. Load or build windows


In [ ]:
# Load prebuilt windows dataset OR build from Sleep-EDF cache
from src.sleep_edf import (
    PROXY_NOTE, STAGE_TO_HEAD_A, find_pilot_pairs,
    load_sleep_edf_recording, windows_with_head_a_labels,
)
from src.head_a import HEAD_A_BINARY_LABELS, labels_to_ids, undersample_balanced, class_weights_from_y, HeadALinear
from src.metrics import macro_f1

WINDOW_SEC, HOP_SEC, TARGET_SR = 2.0, 0.5, 256.0
PRE_SEC, POST_SEC = 20 * 60, 40 * 60

def find_named_dir(name: str):
    root = Path("/kaggle/input")
    cands = []
    if root.exists():
        direct = root / name
        if direct.exists():
            cands.append(direct)
        cands.extend([p for p in root.rglob(name) if p.is_dir()])
    cands.append(WORKING / "exports" / "windows_sc4001")
    cands.append(Path("/workspace/muse-eeg-heads/kaggle_datasets") / name)
    for c in cands:
        if c.exists():
            return c
    return None

win_dir = find_named_dir("muse-eeg-heads-windows")
cache_dir = find_named_dir("muse-eeg-heads-cache")
print("win_dir", win_dir, "cache_dir", cache_dir)

npz_path = None
if win_dir is not None:
    cand = win_dir / "sleep_edf_sc4001_n1slice_windows.npz"
    if cand.exists():
        npz_path = cand

slice_start_sec = None
psg_name, hyp_name = "SC4001E0-PSG.edf", "SC4001EC-Hypnogram.edf"
starts = None

if npz_path is not None and npz_path.exists():
    z = np.load(npz_path, allow_pickle=True)
    X = z["X"].astype(np.float32)
    y = z["y"].astype(np.int64)
    starts = z["starts"] if "starts" in z.files else None
    labels = [HEAD_A_BINARY_LABELS[int(i)] for i in y]
    counts = Counter(labels)
    # try fill slice_start from sibling manifest
    man = win_dir / "manifest.json" if win_dir else None
    if man and man.exists():
        try:
            slice_start_sec = json.loads(man.read_text()).get("slice_start_sec")
        except Exception:
            pass
    print("loaded", npz_path, X.shape, dict(counts))
else:
    if cache_dir is None:
        raise FileNotFoundError("Need muse-eeg-heads-windows npz or muse-eeg-heads-cache")
    cache = cache_dir / "data" / "sleep-edfx-pilot"
    if not cache.exists():
        # sometimes flat
        cache = cache_dir
    pairs = find_pilot_pairs(cache)
    psg, hyp = pairs[0]
    psg_name, hyp_name = psg.name, hyp.name
    rec = load_sleep_edf_recording(
        psg, hyp, target_sr=TARGET_SR, around_stage="stage 1",
        pre_sec=PRE_SEC, post_sec=POST_SEC,
    )
    slice_start_sec = rec["slice_start_sec"]
    Xw, labels, keep = windows_with_head_a_labels(
        rec["data"], rec["stages"], sfreq=TARGET_SR,
        window_sec=WINDOW_SEC, hop_sec=HOP_SEC,
    )
    mask = [lab in HEAD_A_BINARY_LABELS for lab in labels]
    X = Xw[np.asarray(mask)]
    labels = [lab for lab, m in zip(labels, mask) if m]
    y = labels_to_ids(labels, HEAD_A_BINARY_LABELS)
    hop = int(round(HOP_SEC * TARGET_SR))
    starts = np.asarray([keep[i] * hop for i, m in enumerate(mask) if m], dtype=np.int64)
    counts = Counter(labels)
    print("built windows", X.shape, dict(counts))

print("proxy:", PROXY_NOTE)


## 3. Frozen CBraMod + Head A smoke train


In [ ]:
# Frozen encoder + tiny linear head (CPU OK)
from src.cbramod_encoder import FrozenCBraModEncoder

def find_weights():
    root = Path("/kaggle/input")
    names = [
        "models/CBraMod/pretrained_weights.pth",
        "CBraMod/pretrained_weights.pth",
        "pretrained_weights.pth",
    ]
    if root.exists():
        for n in names:
            hits = list(root.rglob(n.split("/")[-1]))
            for h in hits:
                if h.name == "pretrained_weights.pth" and "CBraMod" in str(h):
                    return h
            for h in hits:
                if h.name == "pretrained_weights.pth":
                    return h
    for p in [
        Path("/workspace/muse-eeg-heads/kaggle_datasets/muse-eeg-heads-cache/models/CBraMod/pretrained_weights.pth"),
        Path("/tmp/kaggle_out3/models/CBraMod/pretrained_weights.pth"),
    ]:
        if p.exists():
            return p
    return None

weights = find_weights()
if weights is None:
    raise FileNotFoundError("CBraMod pretrained_weights.pth not found under /kaggle/input")
print("weights", weights)

EPOCHS, BATCH, LR = 3, 32, 1e-3
rng = np.random.default_rng(SEED)
Xb, yb = undersample_balanced(X, y, rng)
print("balanced", Xb.shape, Counter(yb.tolist()))

device = torch.device("cpu")
SKIP = False
notes = {}
history = []
head = None
emb = None
try:
    encoder = FrozenCBraModEncoder(weights, source_sr=TARGET_SR, pool="mean")
    notes = encoder.adapter_notes()
    print("adapter:", notes["fed_input"], "->", notes["native_input"])
    encoder.to(device)
    Xt = torch.from_numpy(Xb)
    emb_list = []
    with torch.no_grad():
        for i in range(0, len(Xt), BATCH):
            emb_list.append(encoder(Xt[i:i+BATCH].to(device)).cpu())
    emb = torch.cat(emb_list, dim=0)
    print("emb", tuple(emb.shape))
except Exception as e:
    print("ENCODER_SKIP:", type(e).__name__, e)
    notes = {"error": str(e)}
    SKIP = True

if not SKIP:
    yt = torch.from_numpy(yb)
    head = HeadALinear(in_dim=emb.shape[-1], n_classes=2).to(device)
    crit = nn.CrossEntropyLoss(weight=class_weights_from_y(yb, n_classes=2))
    opt = torch.optim.Adam(head.parameters(), lr=LR)
    loader = DataLoader(TensorDataset(emb, yt), batch_size=BATCH, shuffle=True)
    for ep in range(EPOCHS):
        head.train()
        total, n = 0.0, 0
        for xb, ybatch in loader:
            opt.zero_grad()
            loss = crit(head(xb), ybatch)
            loss.backward()
            opt.step()
            total += float(loss.item()) * len(ybatch)
            n += len(ybatch)
        head.eval()
        with torch.no_grad():
            pred = head(emb).argmax(-1).numpy()
        row = {
            "epoch": ep + 1,
            "loss": total / max(n, 1),
            "acc": float((pred == yb).mean()),
            "macro_f1": macro_f1(yb.tolist(), pred.tolist(), HEAD_A_BINARY_LABELS),
        }
        history.append(row)
        print(row)
else:
    print("Skipped train — windows/manifest still saved below if possible")


## 4. Export head + provenance manifest


In [ ]:
# Save under /kaggle/working/exports/
EXPORTS = WORKING / "exports"
EXPORTS.mkdir(parents=True, exist_ok=True)

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

win_out = EXPORTS / "sleep_edf_sc4001_n1slice_windows.npz"
np.savez_compressed(
    win_out,
    X=X.astype(np.float32),
    y=y.astype(np.int64),
    starts=starts if starts is not None else np.arange(len(y), dtype=np.int64),
    label_names=np.asarray(HEAD_A_BINARY_LABELS),
)

head_path = EXPORTS / "head_a_binary_state_dict.pt"
if head is not None:
    torch.save(head.state_dict(), head_path)
    print("saved head", head_path)
else:
    print("no head weights (encoder skip)")

manifest = {
    "psg_file": psg_name,
    "hypno_file": hyp_name,
    "slice_start_sec": slice_start_sec,
    "pre_sec": PRE_SEC,
    "post_sec": POST_SEC,
    "window_sec": WINDOW_SEC,
    "hop_sec": HOP_SEC,
    "target_sr": TARGET_SR,
    "channel_proxy_note": PROXY_NOTE,
    "stage_to_label_map": {k: v for k, v in STAGE_TO_HEAD_A.items()},
    "n_windows_per_label": dict(counts),
    "n_windows_total": int(len(y)),
    "random_seed": SEED,
    "encoder_name": "CBraMod",
    "encoder_weights_path": str(weights) if weights else None,
    "encoder_weights_sha256": notes.get("weights_sha256") if isinstance(notes, dict) else None,
    "encoder_adapter": notes,
    "window_shape": list(X.shape),
    "npz_sha256": sha256(win_out),
    "head": "HeadALinear" if head is not None else None,
    "task": "binary_drowsy_vs_hypnagogic",
    "label_list": HEAD_A_BINARY_LABELS,
    "history": history,
    "skipped_encoder": SKIP,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "license_attribution": {
        "Sleep-EDF": "PhysioNet ODC-By",
        "CBraMod": "Apache-2.0 — weighting666/CBraMod + wjq-learning/CBraMod",
    },
}
man_path = EXPORTS / "run_manifest.json"
man_path.write_text(json.dumps(manifest, indent=2))
(EXPORTS / "ATTRIBUTION.txt").write_text(
    "Sleep-EDF Expanded — PhysioNet ODC-By.\n"
    "CBraMod — Apache-2.0 (HF weighting666/CBraMod; code wjq-learning/CBraMod).\n"
    "Publish path: head-only. NO LUNA / L-FAME / SEED-VIG.\n"
)
print("manifest", man_path)
print(json.dumps({k: manifest[k] for k in ("n_windows_per_label", "history", "skipped_encoder")}, indent=2))
